# NYT10m test: DeepSeek Flash with low reasoning

This notebook uses the official manually annotated NYT10m test release. A bag
contains sentences for one **ordered head → tail entity pair**; the output is a set
of supported positive relation IDs, or an empty set (`NA`). Different sentences
may support different relations for the same pair.

The release contains 11,086 rows encoding 9,744 distinct sentence/mention units:
a sentence with several manual labels is repeated once per label. These rows are
merged by exact text, ordered IDs and supplied spans before selecting at most
20 distinct sentence/mention units per bag. Selection is evenly spaced in first
occurrence order and never uses labels or predictions.

The scoring reference is the union of manual positive labels **only for the
selected sentences**. No database facts or omitted-sentence labels are added.
The original file is unchanged. The verified release reconstructs 5,174 bags;
37 bags are shortened, leaving 9,001 selected sentences. In this release the
selected and full bag references happen to coincide (3,899 positive facts).
This coincidence is checked rather than assumed for other datasets.

`nyt10m_test_protocol.py` handles reconstruction, provenance and scoring.
The notebooks contain execution settings and stage calls. Test results do not
select thresholds, prompts, reasoning effort or ensemble rules.


By default, Run All displays the published, fingerprint-checked summaries without local datasets, model weights or paid calls. Enable local analysis or execution explicitly to verify or extend the retained local run.


## 1. Environment and paid-execution controls

Default Run All displays published summaries without an API key or local datasets. `ANALYZE_LOCAL_RESULTS=True` verifies retained local results; `RUN_API=True` enables paid execution or resume on the frozen 5,174-bag scope. Local stages require the project `.venv` kernel. The $10 usage budget includes completed usage and unknown-charge reserves. No off-peak waiting is enabled. Credentials are read only for paid execution from `DEEPSEEK_API_KEY`, the ignored `.env`, or hidden input; they are never written to reports.


In [2]:
# Published report mode does not need an API key, local data or model assets.
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'source/nyt10m_test_protocol.py').is_file())
sys.path.insert(0, str(ROOT / 'source'))
from publication_utils import load_test_report
from nyt10m_test_protocol import scorer_self_check

RUN_API = False
ANALYZE_LOCAL_RESULTS = False
MAX_BUDGET_USD = 10.00
PREVENT_SLEEP = True
# Use exact journal IDs only when explicitly resubmitting an uncertain request.
ALLOW_RESUBMIT_UNCERTAIN_ATTEMPT_IDS = ()
USE_LOCAL_RESULTS = RUN_API or ANALYZE_LOCAL_RESULTS
EXPERIMENT = None
if USE_LOCAL_RESULTS and Path(sys.prefix).resolve() != (ROOT / '.venv').resolve():
    raise RuntimeError('Select the project .venv kernel for local preparation or paid execution.')
print('Mode:', 'local experiment' if USE_LOCAL_RESULTS else 'published reports only')


Mode: published reports only


## 2. Published identity or shared local evidence

Public mode verifies the saved summary against the frozen specification. Local preparation uses the same selected input manifest and manual-reference fingerprint as the SLM notebook. Requests contain supplied names, original selected sentences and character spans, without reference answers or encoder predictions. The prompt, syntax example and parser match the completed v2 validation pilot. Thinking is enabled with low effort and a 4,096-token completion cap including reasoning and final output. No demonstrations or sampling overrides are added.


In [3]:
# Bind the frozen execution settings only for explicitly requested local work.
if USE_LOCAL_RESULTS:
    from nyt10m_test_protocol import prepare_experiment
    import nyt10m_test_api as api
    EXPERIMENT = prepare_experiment()
    CONFIGURATION = api.configure(
        EXPERIMENT, run_api=RUN_API, max_budget_usd=MAX_BUDGET_USD,
        keep_awake=PREVENT_SLEEP,
        allow_resubmit_uncertain_attempt_ids=ALLOW_RESUBMIT_UNCERTAIN_ATTEMPT_IDS)
    display(pd.DataFrame([EXPERIMENT['audit']]))
    display(pd.DataFrame([CONFIGURATION]))
else:
    LLM_REPORT = load_test_report(ROOT, 'test_llm_summary.json')
    print('Published run fingerprint:', LLM_REPORT['fingerprint'])
    print('Local preparation and paid requests disabled.')


Published run fingerprint: b7ab3463602ffb361ff4ecdbc37ce947b0bca7b9cbcfa0221e7761ab3de3f6a3
Local preparation and paid requests disabled.


## 3. Constructed scorer checks and output contract

Synthetic scorer checks use artificial labels without inference. Parser checks additionally run when local preparation is enabled. Malformed JSON, unknown IDs, nested values, repeated object keys, empty content and truncated completions are invalid. Repeated known labels are deduplicated and counted. Invalid responses remain cached without repair or regeneration: positive-label F1 treats them as empty predictions, with a separate invalid rate and no successful exact-match credit for invalid negatives. Accounting uses the tariff settings recorded in the frozen specification.


In [4]:
# Synthetic scoring never submits a request; parser checks use local preparation.
print(scorer_self_check())
if USE_LOCAL_RESULTS:
    assert api.parse_answer('{"relations": []}', 'stop')['status'] == 'valid_empty'
    assert api.parse_answer('{"relations": []}', 'length')['status'] == 'invalid'
else:
    print('Saved reports use the frozen parser and retain invalid-answer counts.')


{'success': True, 'scope': 'synthetic scoring/threshold/voting cases; no model or API calls'}
Saved reports use the frozen parser and retain invalid-answer counts.


## 4. Execute or resume DeepSeek low

Each request is journaled before submission and the response/usage is saved
before its result. Completed valid and invalid outputs are reused. The existing
v2 transport policy retries explicit HTTP 429/502/503/504 at most twice; other
transport uncertainty stops execution before an automatic duplicate submission.
HTTP failures remain pending and can be resumed. Do not delete completed caches
to rerun the scope. Attempt durations and token charges are retained.


In [5]:
# Only RUN_API enables submission. Journal recovery never regenerates invalid answers.
if RUN_API:
    try:
        api.run_experiment()
    except api.RunStopped as exc:
        print('RUN STOPPED:', exc)
    except KeyboardInterrupt:
        print('Interrupted; completed responses are retained. Inspect the attempt ledger before resubmission.')
    except (TimeoutError, ConnectionError, OSError) as exc:
        print('Transport interrupted:', type(exc).__name__, '; inspect the attempt ledger before resubmission.')
else:
    print('Paid API requests disabled.')


Paid API requests disabled.


## 5. Completeness, usage and final low score

Final scoring requires one completed model response for every test bag; invalid
responses are retained, while missing/HTTP requests prevent a final score.
The report contains precision, recall, micro-F1, invalid rate, total/per-bag API
request time and token charges. API timing includes network/service latency and
excludes backoff/idle gaps. Tariff/cache-based charges are reconstructed estimates,
with unknown-charge reserves separate; they are not a provider invoice.


In [6]:
# Never present a partial local run as complete; public mode uses verified exports.
if USE_LOCAL_RESULTS:
    API_STATUS = api.status_and_cost()
    display(pd.DataFrame([API_STATUS]))
    counts = API_STATUS['status_counts']
    if sum(counts.get(k, 0) for k in ['valid', 'valid_empty', 'invalid']) != len(EXPERIMENT['manifest']):
        raise RuntimeError('The local test is incomplete; no final quality score is reported.')
    LLM_REPORT = api.report_api()
else:
    LLM_REPORT = load_test_report(ROOT, 'test_llm_summary.json')
    display(pd.DataFrame([LLM_REPORT['accounting']]))
with pd.option_context('display.max_columns', None):
    display(pd.DataFrame(LLM_REPORT['rows']))
print('Cost note:', LLM_REPORT['cost_note'])
print('Compact report: results/test_llm_summary.json')


,attempts,backend_fingerprints,completed_with_usage,completion_tokens_including_reasoning,expected_bags,peak_rate_attempts,prompt_tokens,request_wall_s_including_attempts,status_counts,token_charge_USD_estimate,unknown_charge_attempts,unknown_charge_reserve_USD
0,5176,[aeb56401ca74e127821c4f9126dcb669],5174,3598426,5174,0,3952505,22023.948629,"{'invalid': 78, 'valid': 2506, 'valid_empty': ...",2.409668,2,0.012103


,FN,FP,TP,bags,invalid,invalid_rate,mean_wall_s,micro_f1,positive_predictions_on_reference_empty_bags,precision,recall,reference_empty_bags,successful_exact_match_bags,system,timing_scope,token_charge_USD_estimate,total_wall_s
0,1721,390,2178,5174,78,0.015075,4.256658,0.673574,270,0.848131,0.558605,1855,3317,deepseek_low,sum of API request attempts; includes service/...,2.409668,22023.948629


Cost note: tariff/cache-based reconstruction, not an invoice; unknown charges remain separate
Compact report: results/test_llm_summary.json


## 6. Matched SLM–ensemble–LLM comparison

This cell reads completed summaries with the same selected evidence and reference.
It does not load models, make requests, tune thresholds or choose a new method.
Both local and API timing scopes remain explicit in the combined report.


In [7]:
# Compare the same frozen systems and reference without new inference.
COMBINED_REPORT = load_test_report(ROOT, 'test_comparison_summary.json')
with pd.option_context('display.max_columns', None):
    display(pd.DataFrame(COMBINED_REPORT['rows']))
print(COMBINED_REPORT['timing_note'])


,FN,FP,TP,bags,invalid,invalid_rate,mean_wall_s,micro_f1,positive_predictions_on_reference_empty_bags,precision,recall,reference_empty_bags,successful_exact_match_bags,system,timing_scope,total_wall_s,token_charge_USD_estimate
0,1973,1543,1926,5174,0,0.000000,0.378337,0.522801,526,0.555203,0.493973,1855,2410,large_0.7,sum of CPU bag attempts plus session label enc...,1957.517133,NaN
1,2366,688,1533,5174,0,0.000000,0.735194,0.500980,308,0.690230,0.393178,1855,2588,majority_0.5_0.7_0.7,sum of CPU bag attempts plus session label enc...,3803.894352,NaN
2,1721,390,2178,5174,78,0.015075,4.256658,0.673574,270,0.848131,0.558605,1855,3317,deepseek_low,sum of API request attempts; includes service/...,22023.948629,2.409668


local CPU member sums and API network/service latency are distinct measurements


## Saved-result statistical analysis

[Test_results_analysis.ipynb](Test_results_analysis.ipynb) contains the completed paired bag-bootstrap F1 intervals, quality–latency and precision/recall figures, relation-level voting diagnostics and separate cost dimensions. It uses saved predictions only; default Run All does not load models or call an API.
